In this notebook, we transform raw, noisy news data into a structured numerical format that a computer can understand.

# Phase 1: Data Scouting & Cleaning

In [42]:
import os
import pandas as pd
import numpy as np
import joblib
import nltk
import re
import string

# Colab Integration: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
    print("✅ Running in Colab. Google Drive mounted.")
except ImportError:
    BASE_PATH = '.'
    print("💻 Running Locally.")

# NLTK Downloads for ephemeral environments
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# Ensure directories exist
os.makedirs(os.path.join(BASE_PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, 'dataset'), exist_ok=True)


💻 Running Locally.


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/basstianlopez/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/basstianlopez/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/basstianlopez/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 1. Data Ingestion
Load our primary dataset. It contains nearly 40,000 articles, balanced almost 50/50 between Real (1) and Fake (0) news.

In [43]:

data_path = os.path.join(BASE_PATH, 'dataset/data.csv')
df = pd.read_csv(data_path)
df.head(5)

,label,title,text,subject,date
0,1,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,1,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,1,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,1,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


### 1.1 Label Normalization & Class Balancing

In [44]:
# DOC REF: summary/phase-1.md -> 4. Label Normalization & Class Balancing (1.3)
# Random Undersampling to resolve Class Imbalance (Perfect 50/50 Split)
# -------------------------------------------------------------------------------------
# 1. Forensic Audit: Initial State
print("INITIAL CLASS DISTRIBUTION (Raw Data):")
print(df['label'].value_counts())
print(f"Total specimens: {len(df)}")

# 2. Resampling Execution
min_specimens = df['label'].value_counts().min()
df = df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min_specimens, 
random_state=42)).reset_index(drop=True)

# 3. Validation: Final State
print(f"\nDATASET BALANCED (undersampled to {min_specimens} per class).")
print("\nAUDIT:")
print(df['label'].value_counts())
print(f"Unique Label Values (Verification): {df['label'].unique()}")
print(f"Ready for cleaning: {len(df)} specimens.")

INITIAL CLASS DISTRIBUTION (Raw Data):
label
1    19999
0    19943
Name: count, dtype: int64
Total specimens: 39942

DATASET BALANCED (undersampled to 19943 per class).

AUDIT:
label
0    19943
1    19943
Name: count, dtype: int64
Unique Label Values (Verification): [0 1]
Ready for cleaning: 39886 specimens.


/var/folders/dk/w5mlpx9x4t9fj30jk64759lh0000gn/T/ipykernel_30673/1511271236.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min_specimens,


## 2. Fusion (Title + Text)

The Strategy: 
Created a new column `full_text` by concatenating `title` and `text`.

In [45]:
# DOC REF: summary/phase-1.md -> 2. Data Segmentation & Feature Construction (1.2) 
# Maximize context density (merging headline hook + body content)
# -------------------------------------------------------------------------------------

print(f"Original Column Count: {df.shape[1]}")
# Fusing the text features
df['full_text'] = df['title'] + " " + df['text']
print(f"Updated Column Count: {df.shape[1]}")
# Forensic Demonstration of the Fusion (Using index 0 as proof)

print("\nTITLE (The Hook):")
print(f"   {df['title'].iloc[0]}")
print("\nBODY (Content snippet):")
# Limiting output to the first 100 characters to prevent console clutter
print(f"   {df['text'].iloc[0][:100]}...") 
print("\nFULL TEXT (Combined hook + snippet):")
print(f"   {df['full_text'].iloc[0][:150]}...")
print("\n")
# Displaying a clean view of the final target variable and the newly engineered feature
df[['label', 'full_text']].head(5)

Original Column Count: 5
Updated Column Count: 6

TITLE (The Hook):
   THIRD GRADE BOYS COMPLAIN About 9 Year Old Girl Using Boys’ Bathroom…Boys Told To Stand Closer To Urinals

BODY (Content snippet):
   Girls aren t the only gender who will suffer embarrassment, humiliation and even the prospect of sex...

FULL TEXT (Combined hook + snippet):
   THIRD GRADE BOYS COMPLAIN About 9 Year Old Girl Using Boys’ Bathroom…Boys Told To Stand Closer To Urinals Girls aren t the only gender who will suffer...




,label,full_text
0,0,THIRD GRADE BOYS COMPLAIN About 9 Year Old Gir...
1,0,Mitch McConnell TRASHES Trump: It’s ‘Obvious’...
2,0,"Supreme Court Gives A Big ‘F You’ To GOP, Rej..."
3,0,The Panama Papers: How The 1%’s Greed Has Lit...
4,0,(VIDEO) REV AL SHARPTON BOTCHES THE NAME OF A ...


In [46]:
df['full_text'] = df['title'] + " " + df['text']
print("'full_text' Column created.")
df.head(5)

'full_text' Column created.


,label,title,text,subject,date,full_text
0,0,THIRD GRADE BOYS COMPLAIN About 9 Year Old Gir...,Girls aren t the only gender who will suffer e...,politics,"May 26, 2016",THIRD GRADE BOYS COMPLAIN About 9 Year Old Gir...
1,0,Mitch McConnell TRASHES Trump: It’s ‘Obvious’...,Many Republicans have expressed the opinion th...,News,"June 10, 2016",Mitch McConnell TRASHES Trump: It’s ‘Obvious’...
2,0,"Supreme Court Gives A Big ‘F You’ To GOP, Rej...",The Supreme Court did something amazing on Mon...,News,"May 23, 2016","Supreme Court Gives A Big ‘F You’ To GOP, Rej..."
3,0,The Panama Papers: How The 1%’s Greed Has Lit...,One of the immediate benefits of The Panama Pa...,News,"April 4, 2016",The Panama Papers: How The 1%’s Greed Has Lit...
4,0,(VIDEO) REV AL SHARPTON BOTCHES THE NAME OF A ...,REVEREND AL MIGHT WANT TO GO BACK TO SCHOOL OR...,Government News,"Jul 24, 2015",(VIDEO) REV AL SHARPTON BOTCHES THE NAME OF A ...


## 3. The Scrubbing (Cleaning Logic)

In [47]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def clean_text(text):
    
    # Standard Cleaning logic: 1.Stemming > 2.Tokenization > 3.Lemmatization > 4.Segmentation
    
    if pd.isna(text):
        return ""

    # 1. Normalization > Lowercase 
    text = text.lower()

    # 2. Regex Cleaning & Standardizing (Removing non-alphanumeric noise & punctuation)
    pattern = re.compile('[%s]' % re.escape(string.punctuation))
    text = pattern.sub('', text)
    # 3. Tokenization (Segmenting strings into individual semantic units)
    tokens = word_tokenize(text)
    # 4. Lemmatization/Stemming (Conceptual Layer / Baseline limitation note)

    # *Note: Focus here remains on stopword removal for the baseline's 'Peeling' phase.
    
    # 5. Stopword Removal (Eliminating high-frequency, low-variance noise)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [w for w in tokens if w not in stop_words]

    return " ".join(filtered_tokens)


In [48]:
# DOC REF: summary/phase-1.md -> 3. Pre-Processing Pipeline Execution
# -------------------------------------------------------------------------------------
import time

start_time = time.time()

# Apply the cleaning pipeline
df['cleaned_text'] = df['full_text'].apply(clean_text)
execution_time = time.time() - start_time
print(f"Scrubbing Complete in {execution_time:.2f} seconds.\n")

# Forensic Demonstration of Scrubbing (Using index 0 as proof)

print("RAW TEXT (Before Pipeline):")
print(f"   {df['full_text'].iloc[0][:150]}...\n")
print("\nCLEANED TEXT (After Pipeline):")

# Notice how the punctuation is gone, it's lowercased, and words like 'As', 'U.S.', 'The' are processed.
print(f"   {df['cleaned_text'].iloc[0][:150]}...")

Scrubbing Complete in 62.97 seconds.

RAW TEXT (Before Pipeline):
   THIRD GRADE BOYS COMPLAIN About 9 Year Old Girl Using Boys’ Bathroom…Boys Told To Stand Closer To Urinals Girls aren t the only gender who will suffer...


CLEANED TEXT (After Pipeline):
   third grade boys complain 9 year old girl using boys ’ bathroom…boys told stand closer urinals girls gender suffer embarrassment humiliation even pros...


## 4. Vectorization (TF-IDF / ENBEDINGS)

In [49]:
# DOC REF: summary/phase-1.md -> 5. NLP Feature Extraction: My Baseline Vectorization (1.4)
# Mechanism: TfidfVectorizer to convert tokens into numerical indices.
# -------------------------------------------------------------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer

# Initializing the Vectorizer with the exact hyperparameters from the summary
# Limit to 5000 features to maintain transparency and avoid overfitting
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

print(f"Max Features: {tfidf.max_features} (Prevents noise and model overfitting)")
print(f"\nN-Grams:      {tfidf.ngram_range} (Captures multi-word context like 'White House')")

# *Note: The actual fitting of this vectorizer happens after we split the data (next step) 
# to avoid "data leakage" from the testing set into our baseline engine.


Max Features: 5000 (Prevents noise and model overfitting)

N-Grams:      (1, 2) (Captures multi-word context like 'White House')


## 5. The Final Slice (Splitting the Data)

In [50]:
# DOC REF: summary/phase-1.md -> 6. Dataset Segmentation (1.5)
# Technical Guard: Stratified 80/20 Split to maintain label distribution.
# -------------------------------------------------------------------------------------
from sklearn.model_selection import train_test_split
# Performing the split (80% Train / 20% Test)
train_df, test_df = train_test_split(
    df, 
    test_size=0.20, 
    random_state=42, 
    stratify=df['label']
)

print(f"Training Set: {len(train_df)} specimens (My development environment)")
print(f"Testing Set:  {len(test_df)} specimens (My validation 'Final Exam')")

# PROVING STRATIFICATION: Comparing distributions
train_dist = train_df['label'].value_counts(normalize=True).round(4) * 100
test_dist = test_df['label'].value_counts(normalize=True).round(4) * 100
print(f"Train Set Distribution: Fake={train_dist[0]}%, Real={train_dist[1]}%")
print(f"Test Set Distribution:  Fake={test_dist[0]}%, Real={test_dist[1]}%")

Training Set: 31908 specimens (My development environment)
Testing Set:  7978 specimens (My validation 'Final Exam')
Train Set Distribution: Fake=50.0%, Real=50.0%
Test Set Distribution:  Fake=50.0%, Real=50.0%


## 6. Persisting Data & Preparing Ingredients
We save our progress to ensure Phase 2 starts with all necessary ingredients ready.


In [51]:
# DOC REF: summary/phase-1.md -> 🏛️ My Technical Knowledge: Serialization & Reliability
# Using Joblib for Pipeline Decoupling.
# -------------------------------------------------------------------------------------
import os
import joblib

# 1. The Translator Engine (Vectorizer Persistence)
#    Important: We FIT the vectorizer ONLY on the Training set to prevent data leakage.
tfidf.fit(train_df['cleaned_text'].astype(str))

# Saving the 'Frozen' Dictionary/Vocabulary
vectorizer_path = os.path.join(BASE_PATH, 'models/vectorizer.joblib')
joblib.dump(tfidf, vectorizer_path)
print(f"'vectorizer.joblib' saved to {vectorizer_path}")
print("   *Note: This 'frozen' vocabulary ensures identical indices for Phase 2 & 3 inference.")

# 2. Pipeline Decoupling: Saving processed datasets for next phase
train_df.to_csv(os.path.join(BASE_PATH, 'dataset/train.csv'), index=False)
test_df.to_csv(os.path.join(BASE_PATH, 'dataset/test.csv'), index=False)
df.to_csv(os.path.join(BASE_PATH, 'dataset/cleaned_data.csv'), index=False)


'vectorizer.joblib' saved to ./models/vectorizer.joblib
   *Note: This 'frozen' vocabulary ensures identical indices for Phase 2 & 3 inference.
